In [5]:
import requests
import os
import dotenv
from dotenv import load_dotenv

In [6]:
load_dotenv()  # Load environment variables from .env file

True

In [ ]:
EDESK_KEY = os.getenv("EDESK_KEY")  


In [29]:
headers = {"Authorization": EDESK_KEY}
url = "https://api.edesk.com/v1/tickets?filter_status_equals=Closed"
response = requests.get(
    url,
    headers=headers
)

In [32]:
print(response.json()["data"][1])

{'id': 680474220, 'subject': 'Questions diverses', 'channel_id': 316593, 'status': 'Closed', 'type': 'OrderQuery', 'sales_order_id': 3177965967, 'sales_order': {'id': 3177965967, 'channel_id': 316593, 'status': 'Delivered', 'seller_order_id': 'F905JA03728-A', 'created_at': '2024-11-23 19:22:22', 'contact_id': 3090461865, 'total_amount': 1619.9, 'shipping_amount': 0, 'tracking_codes': [{'tracking_code': "'1Z62369V6826164776'", 'tracking_link': 'https://wwwapps.ups.com/WebTracking/track?track=yes&trackNums=1Z62369V6826164776', 'tracking_carrier_name': 'UPS'}], 'last_updated_at': '2024-11-28 12:27:32', 'order_items': [{'id': 3320019007, 'quantity': 1, 'end_price': 1619.9, 'product': {'id': 223379627, 'sku': 9000714214, 'title': 'Sedatech PC Gamer Pro Watercooling Full', 'brand': None, 'currency': '€', 'dimensions': None, 'weight': 453.59237, 'product_image_url': 'https://merchant.boulanger.com/media/product/image/0fbc356d-0304-436b-b5d9-64391ed39b86', 'price': 1619.9, 'marketplace_link': 

In [119]:
import sqlite3, json
conn = sqlite3.connect("tickets.db")

no_customer = 0
no_agent = 0
empty_after_join = 0

tickets = conn.execute("SELECT ticket_id FROM tickets").fetchall()
for (ticket_id,) in tickets:
    messages = conn.execute(
        "SELECT sender_role, body FROM messages WHERE ticket_id = ?", (ticket_id,)
    ).fetchall()
    customer_msgs = [m for m in messages if m[0] == "Customer"]
    agent_msgs = [m for m in messages if m[0] == "Agent"]
    if not customer_msgs:
        no_customer += 1
    elif not agent_msgs:
        no_agent += 1
    elif not "\n".join(b for r, b in customer_msgs if b).strip():
        empty_after_join += 1

print("no customer messages:", no_customer)
print("no agent messages:", no_agent)
print("empty after join:", empty_after_join)

no customer messages: 2631
no agent messages: 2143
empty after join: 6


In [121]:

import sqlite3
cnx = sqlite3.connect('tickets.db')
cursor = cnx.cursor()
cursor.execute("SELECT DISTINCT sender_role FROM messages;")
print(cursor.fetchall())
cnx.close()

[('Customer',), ('Agent',)]


In [118]:

import sqlite3
cnx = sqlite3.connect('tickets.db')
cursor = cnx.cursor()
cursor.execute("UPDATE tickets SET embedded = 0 WHERE embedded = 1;")
cnx.commit()
print(cursor.fetchall())
cnx.close()

[]


In [ ]:
headers = {"Authorization": EDESK_KEY}
url = "https://api.edesk.com/v1/tickets/680474220?include=messages"
response = requests.get(
    url,
    headers=headers
)


In [100]:
import sqlite3
conn = sqlite3.connect("tickets.db")

# How many tickets have zero messages vs at least one?
print("tickets with messages:", conn.execute("""
    SELECT COUNT(DISTINCT ticket_id) FROM messages
""").fetchone())

print("total tickets:", conn.execute("SELECT COUNT(*) FROM tickets").fetchone())

# Which tickets are missing messages entirely?
missing = conn.execute("""
    SELECT t.ticket_id FROM tickets t
    LEFT JOIN messages m ON t.ticket_id = m.ticket_id
    WHERE m.ticket_id IS NULL
    LIMIT 10
""").fetchall()
print("sample tickets with no messages:", missing)

tickets with messages: (8041,)
total tickets: (10000,)
sample tickets with no messages: [(496665059,), (497363785,), (498325972,), (498385835,), (498625451,), (498928390,), (500593989,), (501434122,), (501622197,), (501912072,)]


In [43]:
response.json()["data"]["messages_ids"]

[3415493292,
 3415493296,
 3415493310,
 3417183677,
 3417229997,
 3417230016,
 3563209849,
 3563209851,
 3563792340,
 3563937977,
 3563937983]

In [55]:
headers = {"Authorization": EDESK_KEY}
url = "https://api.edesk.com/v1/messages/3563937983"
response = requests.get(
    url,
    headers=headers
)
response.json()["data"]

{'id': 3563937983,
 'external_id': None,
 'subject': None,
 'direction': 'Other',
 'from_user': {'id': 174060,
  'name': 'Loic DUBS',
  'email': 'loic.dubs@sedatech.de',
  'active': 1,
  'username': '6479b725f32c6',
  'role': None},
 'type': 'Message',
 'body': False,
 'from_consumer_id': None,
 'attachments': None,
 'errors': None,
 'created_at': 1783418645,
 'ticket_id': 680474220}

In [63]:
import json
tickets = json.load(open("data/tickets.json"))

In [70]:
tickets[1111]["messages"][0]

{'is_incoming': '1',
 'created_at': '2025-08-11 13:53:11',
 'message_body': 'Thank you for delivering the computer for order No. 305-2577694-0095527 via Amazon.de. We are very satisfied with the build quality and configuration, and it would be a great pity if we had to return the product due to an invoicing error.',
 'type': 'Consumer Conversation',
 'language': 'English (UK)',
 'delivery_status': 'pending delivery',
 'response_time': 0,
 'handling_time': 0}